In [ ]:
# 공통 경로와 패키지
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta
import numpy as np

table_dir = str(PROCESSED_DIR / 'AEGIS-Dx-SNOMED/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제

columns_to_drop = [
    'dig_memo', 'pt_dig_memo'
]

for file in csv_files:
    try:
        snomed_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        total_cols_to_remove = len(columns_to_drop)
        present_columns = [col for col in columns_to_drop if col in snomed_diagnoses_df.columns]
        removed_cols_count = len(present_columns)

        if removed_cols_count > 0:
            snomed_diagnoses_df.drop(columns = present_columns, inplace = True)

        snomed_diagnoses_df.to_csv(file, index=False, encoding = 'utf-8')

    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# opd_nm, depart, ad_ward에서 신부인과, 정신건강의학과 제거

import os
import glob
import pandas as pd

# 키워드 정의
keywords = "산부인과|정신건강의학과|정신의학과"

# 처리 시작
for file in csv_files:
    try:
        snomed_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        original_len = len(snomed_diagnoses_df)

        # 각 컬럼에 대해 조건에 맞는 행 제거
        condition = pd.Series([False] * len(snomed_diagnoses_df))
        for col in ['depart', 'ad_depart_nm', 'dc_depart_nm']:
            if col in snomed_diagnoses_df.columns:
                condition = condition | snomed_diagnoses_df[col].fillna('').str.contains(keywords, regex=True)

        # 해당 행 제거
        snomed_diagnoses_df = snomed_diagnoses_df[~condition]
        removed = original_len - len(snomed_diagnoses_df)

        # 덮어쓰기 저장
        snomed_diagnoses_df.to_csv(file, index=False, encoding='utf-8')

    except Exception as e:
        print(f"[ERROR] 오류 발생 - {file}: {e}")


In [ ]:
# pt_id 치환
for file in csv_files:
    try:
        snomed_diagnoses_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in snomed_diagnoses_df.columns:
            snomed_diagnoses_df['pt_id'] = snomed_diagnoses_df['pt_id'].map(patient_id_map).fillna(snomed_diagnoses_df['pt_id'])

        snomed_diagnoses_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# hos_dt ±7일 ±30분
for file in csv_files:
    try:
        snomed_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'hos_dt' in snomed_diagnoses_df.columns:
            snomed_diagnoses_df['hos_dt'] = pd.to_datetime(snomed_diagnoses_df['hos_dt'], errors='coerce')
            snomed_diagnoses_df['hos_dt'] = snomed_diagnoses_df['hos_dt'] + snomed_diagnoses_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            snomed_diagnoses_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# tx_dt ±7일 ±30분
for file in csv_files:
    try:
        snomed_diagnoses_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'tx_dt' in snomed_diagnoses_df.columns:
            snomed_diagnoses_df['tx_dt'] = pd.to_datetime(snomed_diagnoses_df['tx_dt'], errors='coerce')
            snomed_diagnoses_df['tx_dt'] = snomed_diagnoses_df['tx_dt'] + snomed_diagnoses_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            snomed_diagnoses_df.to_csv(file, index=False, encoding = 'utf-8')
    except Exception as e:
        print(f"[ERROR] tx_dt 오류: {file} → {e}")
